# Cryptography Lab
**Set E1: Classical Ciphers & Number Theory | Set E2: Modern Crypto**

## E1-1: Caesar Cipher

In [1]:
def caesar_encrypt(text, shift):
    result = ""
    for c in text:
        if c.isalpha():
            base = 65 if c.isupper() else 97
            result += chr((ord(c) - base + shift) % 26 + base)
        else:
            result += c
    return result

def caesar_decrypt(text, shift):
    return caesar_encrypt(text, -shift)

msg = "Hello World"
enc = caesar_encrypt(msg, 3)
dec = caesar_decrypt(enc, 3)
print("Original:", msg)
print("Encrypted:", enc)
print("Decrypted:", dec)


Original: Hello World
Encrypted: Khoor Zruog
Decrypted: Hello World


## E1-2: One-Time Pad

In [ ]:
import random, string

def generate_key(length):
    return ''.join(random.choice(string.ascii_uppercase) for _ in range(length))

def otp_encrypt(text, key):
    return ''.join(chr((ord(t) - 65 + ord(k) - 65) % 26 + 65) for t, k in zip(text.upper(), key))

def otp_decrypt(cipher, key):
    return ''.join(chr((ord(c) - ord(k)) % 26 + 65) for c, k in zip(cipher, key))

msg = "HELLO"
key = generate_key(len(msg))
enc = otp_encrypt(msg, key)
dec = otp_decrypt(enc, key)
print("Key:", key)
print("Encrypted:", enc)
print("Decrypted:", dec)


## E1-3: Monoalphabetic Substitution Cipher

In [ ]:
import random, string

alphabet = list(string.ascii_uppercase)
shuffled = alphabet.copy()
random.shuffle(shuffled)
enc_map = dict(zip(alphabet, shuffled))
dec_map = dict(zip(shuffled, alphabet))

def mono_encrypt(text):
    return ''.join(enc_map.get(c, c) for c in text.upper())

def mono_decrypt(text):
    return ''.join(dec_map.get(c, c) for c in text.upper())

msg = "HELLO WORLD"
enc = mono_encrypt(msg)
dec = mono_decrypt(enc)
print("Key map:", enc_map)
print("Encrypted:", enc)
print("Decrypted:", dec)


## E1-4: GCD using Euclidean Algorithm

In [ ]:
def gcd(a, b):
    while b:
        a, b = b, a % b
    return a

print("GCD(48, 18) =", gcd(48, 18))


## E1-5: Caesar Cipher Brute-Force Attack

In [ ]:
def caesar_bruteforce(cipher):
    for shift in range(26):
        print(shift, ":", caesar_decrypt(cipher, shift))

cipher = caesar_encrypt("ATTACK AT DAWN", 7)
print("Ciphertext:", cipher)
print()
caesar_bruteforce(cipher)


## E1-6: Playfair Cipher

In [ ]:
def playfair_matrix(key):
    key = key.upper().replace("J", "I")
    seen = set()
    matrix = []
    for c in key + string.ascii_uppercase:
        if c not in seen and c != 'J':
            seen.add(c)
            matrix.append(c)
    return [matrix[i*5:i*5+5] for i in range(5)]

def find_pos(matrix, c):
    for i, row in enumerate(matrix):
        if c in row:
            return i, row.index(c)

def prepare_text(text):
    text = text.upper().replace("J", "I").replace(" ", "")
    pairs = []
    i = 0
    while i < len(text):
        a = text[i]
        b = text[i+1] if i+1 < len(text) else 'X'
        if a == b:
            pairs.append(a + 'X')
            i += 1
        else:
            pairs.append(a + b)
            i += 2
    return pairs

def playfair_encrypt(text, key):
    matrix = playfair_matrix(key)
    pairs = prepare_text(text)
    result = ""
    for pair in pairs:
        r1, c1 = find_pos(matrix, pair[0])
        r2, c2 = find_pos(matrix, pair[1])
        if r1 == r2:
            result += matrix[r1][(c1+1)%5] + matrix[r2][(c2+1)%5]
        elif c1 == c2:
            result += matrix[(r1+1)%5][c1] + matrix[(r2+1)%5][c2]
        else:
            result += matrix[r1][c2] + matrix[r2][c1]
    return result

def playfair_decrypt(cipher, key):
    matrix = playfair_matrix(key)
    result = ""
    for i in range(0, len(cipher), 2):
        a, b = cipher[i], cipher[i+1]
        r1, c1 = find_pos(matrix, a)
        r2, c2 = find_pos(matrix, b)
        if r1 == r2:
            result += matrix[r1][(c1-1)%5] + matrix[r2][(c2-1)%5]
        elif c1 == c2:
            result += matrix[(r1-1)%5][c1] + matrix[(r2-1)%5][c2]
        else:
            result += matrix[r1][c2] + matrix[r2][c1]
    return result

key = "MONARCHY"
msg = "INSTRUMENTS"
enc = playfair_encrypt(msg, key)
dec = playfair_decrypt(enc, key)
print("Key:", key)
print("Encrypted:", enc)
print("Decrypted:", dec)


## E1-7: 2x2 Hill Cipher

In [ ]:
import numpy as np

def mod_inverse(a, m):
    a = a % m
    for x in range(1, m):
        if (a * x) % m == 1:
            return x
    return None

def hill_encrypt(text, key_matrix):
    text = text.upper().replace(" ", "")
    if len(text) % 2 != 0:
        text += 'X'
    result = ""
    for i in range(0, len(text), 2):
        pair = np.array([ord(text[i])-65, ord(text[i+1])-65])
        enc = np.dot(key_matrix, pair) % 26
        result += chr(enc[0]+65) + chr(enc[1]+65)
    return result

def hill_decrypt(cipher, key_matrix):
    det = int(round(np.linalg.det(key_matrix)))
    det_inv = mod_inverse(det, 26)
    adj = np.array([[key_matrix[1][1], -key_matrix[0][1]],
                     [-key_matrix[1][0], key_matrix[0][0]]])
    inv_matrix = (det_inv * adj) % 26
    result = ""
    for i in range(0, len(cipher), 2):
        pair = np.array([ord(cipher[i])-65, ord(cipher[i+1])-65])
        dec = np.dot(inv_matrix, pair) % 26
        result += chr(int(dec[0])+65) + chr(int(dec[1])+65)
    return result

key_matrix = np.array([[3, 3], [2, 5]])
msg = "HELP"
enc = hill_encrypt(msg, key_matrix)
dec = hill_decrypt(enc, key_matrix)
print("Encrypted:", enc)
print("Decrypted:", dec)


## E1-8: Transposition Cipher (Double Encryption)

In [ ]:
def transposition_encrypt(text, key):
    text = text.replace(" ", "")
    cols = len(key)
    rows = -(-len(text) // cols)
    padded = text.ljust(rows*cols, 'X')
    grid = [padded[i:i+cols] for i in range(0, len(padded), cols)]
    order = sorted(range(cols), key=lambda k: key[k])
    result = ""
    for col in order:
        for row in grid:
            result += row[col]
    return result

msg = "MEETMEAFTERPARTY"
key = "ZEBRA"
once = transposition_encrypt(msg, key)
twice = transposition_encrypt(once, key)
print("Plaintext:", msg)
print("Single encryption:", once)
print("Double encryption:", twice)


## E1-9: ElGamal Cryptosystem

In [ ]:
import random

def find_primitive_root(p):
    for g in range(2, p):
        seen = set(pow(g, i, p) for i in range(1, p))
        if len(seen) == p - 1:
            return g
    return None

p = 23
g = find_primitive_root(p)
x = random.randint(1, p-2)   # private key
y = pow(g, x, p)             # public key

def elgamal_encrypt(m, p, g, y):
    k = random.randint(1, p-2)
    c1 = pow(g, k, p)
    c2 = (m * pow(y, k, p)) % p
    return c1, c2

def elgamal_decrypt(c1, c2, x, p):
    s = pow(c1, x, p)
    s_inv = pow(s, p-2, p)
    return (c2 * s_inv) % p

m = 15
c1, c2 = elgamal_encrypt(m, p, g, y)
dec = elgamal_decrypt(c1, c2, x, p)
print(f"p={p}, g={g}, private x={x}, public y={y}")
print("Message:", m)
print("Ciphertext:", (c1, c2))
print("Decrypted:", dec)


---
# Set E2: DES, RSA, Diffie-Hellman

## E2-1: DES Encryption/Decryption
*(requires `pycryptodome`: `pip install pycryptodome`)*

In [ ]:
from Crypto.Cipher import DES
from Crypto.Util.Padding import pad, unpad

key = b"8bytekey"   # DES key must be exactly 8 bytes
cipher = DES.new(key, DES.MODE_ECB) # ECB → CBC → CFB → OFB → CTR

msg = b"Hello DES!"
padded = pad(msg, DES.block_size)
enc = cipher.encrypt(padded)
dec = unpad(cipher.decrypt(enc), DES.block_size)

print("Original:", msg.decode())
print("Encrypted (hex):", enc.hex())
print("Decrypted:", dec.decode())


Original: Hello DES!
Encrypted (hex): 8303e1c88556debb8812abe12232fd70
Decrypted: Hello DES!


In [12]:
from Crypto.Cipher import DES
from Crypto.Util.Padding import pad, unpad

# Take input from user
message = input("Enter your message: ")
key = input("Enter 8-byte key: ")

# Check key length
if len(key) != 8:
    print("Error: DES key must be exactly 8 characters!")
    exit()

# Convert key and message into bytes
key = key.encode()
msg = message.encode()

# ---------------- ENCRYPTION ----------------

cipher = DES.new(key, DES.MODE_ECB)

padded = pad(msg, DES.block_size)

enc = cipher.encrypt(padded)

print("\nOriginal:", message)
print("Encrypted (hex):", enc.hex())


# ---------------- DECRYPTION ----------------

decipher = DES.new(key, DES.MODE_ECB)

decrypted = decipher.decrypt(enc)

dec = unpad(decrypted, DES.block_size)

print("Decrypted:", dec.decode())


Original: RIfat Hossen
Encrypted (hex): c635543d3cbaf57e1649e2ff44e893d1
Decrypted: RIfat Hossen


## E2-2: RSA Algorithm

In [ ]:
def rsa_keygen():
    p=int(input("Enter the 1st Prime Number: "))
    q=int(input("Enter the 2nd Prime Number: "))
    n = p * q
    phi = (p-1) * (q-1)
    e = int(input("Enter the public exponent e (must be coprime to φ(n)): "))
    d = pow(e, -1, phi)
    return (e, n), (d, n)   # public, private

pub, priv = rsa_keygen()

def rsa_encrypt(m, pub):
    e, n = pub
    return pow(m, e, n)

def rsa_decrypt(c, priv):
    d, n = priv
    return pow(c, d, n)

m = 65
c = rsa_encrypt(m, pub)
dec = rsa_decrypt(c, priv)
print("Public key:", pub)
print("Private key:", priv)
print("Message:", m, "| Encrypted:", c, "| Decrypted:", dec)


Public key: (17, 3965)
Private key: (2033, 3965)
Message: 65 | Encrypted: 3705 | Decrypted: 65


## E2-3: Diffie-Hellman Key Exchange

In [ ]:
import random

p = 23   # public prime
g = 5    # public primitive root

a = random.randint(1, p-2)   # Alice private
b = random.randint(1, p-2)   # Bob private

A = pow(g, a, p)   # Alice public
B = pow(g, b, p)   # Bob public

shared_A = pow(B, a, p)
shared_B = pow(A, b, p)

print("Alice private:", a, "| Bob private:", b)
print("Alice public:", A, "| Bob public:", B)
print("Alice's shared key:", shared_A)
print("Bob's shared key:  ", shared_B)
print("Keys match:", shared_A == shared_B)


## E2-4: Modular Arithmetic using a Primitive Root

In [ ]:
def find_primitive_root(p):
    for g in range(2, p):
        seen = set(pow(g, i, p) for i in range(1, p))
        if len(seen) == p - 1:
            return g
    return None

p = 23
g = find_primitive_root(p)
print(f"Primitive root of {p} is {g}\n")
for i in range(1, p):
    print(f"{g}^{i} mod {p} = {pow(g, i, p)}")


## E2-5: Extended Euclidean Algorithm

In [2]:
def extended_gcd(a, b):
    if b == 0:
        return a, 1, 0
    g, x1, y1 = extended_gcd(b, a % b)
    x = y1
    y = x1 - (a // b) * y1
    return g, x, y

a, b = 56, 15
g, x, y = extended_gcd(a, b)
print(f"GCD({a},{b}) = {g}")
print(f"x = {x}, y = {y}")
print(f"Check: {a}*{x} + {b}*{y} = {a*x + b*y}")


GCD(56,15) = 1
x = -4, y = 15
Check: 56*-4 + 15*15 = 1
